# 리뷰 분석 모델 학습 — KLUE-BERT 2개 (논문 3.3.3)

| 모델 | 할 일 | 출력 → DB |
|---|---|---|
| ① 노이즈 분류 | 정상 / 노이즈 (이진) | 정상일 확률 → `review.reliability` (0.3 미만이면 제외) |
| ② 부정 유형 분류 | 맛·가격·서비스·위생·대기시간·분위기 (멀티라벨, 시그모이드) | 유형별 확률 → `neg_probs`, 0.5 이상 → `neg_types` |

- **사전학습 모델:** `klue/bert-base`
- **학습:** `train_reviews.csv` — 평가셋과 **다른 표현**으로 작성한 합성 리뷰 4,279건 (평가 문장과 똑같은 문장 0개)
- **평가:** `test_reviews.csv` — 분석에 쓰는 더미 리뷰 303건(정답 포함). 학습에 쓰지 않음
- **비교:** TF-IDF+SVM 베이스라인, 규칙(rule-v1)

**순서:** `런타임 → 런타임 유형 변경 → T4 GPU` → 위에서부터 Shift+Enter. 두 모델 학습 합쳐 약 10분.

In [ ]:
# 1. GPU 확인 + 라이브러리
!nvidia-smi -L
!pip -q install -U transformers datasets accelerate
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| GPU", torch.cuda.is_available())

In [ ]:
# 2. 데이터 올리기: train_reviews.csv, test_reviews.csv (버튼이 나오면 두 파일을 같이 고르기)
import os
import numpy as np, pandas as pd
from google.colab import files
if not (os.path.exists("train_reviews.csv") and os.path.exists("test_reviews.csv")):
    files.upload()
TYPES = ["맛", "가격", "서비스", "위생", "대기시간", "분위기"]
train = pd.read_csv("train_reviews.csv")
test = pd.read_csv("test_reviews.csv")
test_text = test[test.copy_noise == 0].reset_index(drop=True)   # 복붙 중복은 규칙(작성자+본문)이 처리
test_clean = test_text[test_text.is_noise == 0].reset_index(drop=True)
train_clean = train[train.is_noise == 0].reset_index(drop=True)
print("학습", len(train), "건 (노이즈", int(train.is_noise.sum()), ") | 평가", len(test_text), "건 (노이즈", int(test_text.is_noise.sum()), ")")
print("부정 유형 학습", train_clean[TYPES].sum().to_dict())
print("부정 유형 평가", test_clean[TYPES].sum().to_dict())

In [ ]:
# 3. 공통 준비
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments,
                          DataCollatorWithPadding)
BASE = "klue/bert-base"
MAX_LEN = 128
tok = AutoTokenizer.from_pretrained(BASE)
collator = DataCollatorWithPadding(tok)

def to_ds(df, label_col=None, multi=False):
    d = {"text": df.text.tolist()}
    if multi:
        d["labels"] = df[TYPES].astype(float).values.tolist()
    else:
        d["labels"] = df[label_col].astype(int).tolist()
    return Dataset.from_dict(d).map(lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN), batched=True,
                                    remove_columns=["text"])

def args(out, epochs):
    return TrainingArguments(output_dir=out, num_train_epochs=epochs, learning_rate=3e-5,
                             per_device_train_batch_size=16, per_device_eval_batch_size=64, warmup_ratio=0.1,
                             weight_decay=0.01, fp16=torch.cuda.is_available(), logging_steps=50,
                             eval_strategy="no", save_strategy="no", report_to=[], seed=42)

def predict_probs(model, texts, multi=False, bs=64):
    model.eval(); out = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i + bs], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(model.device)
        with torch.no_grad():
            logits = model(**enc).logits
        out.append((torch.sigmoid(logits) if multi else torch.softmax(logits, -1)).cpu().numpy())
    return np.concatenate(out)

## ① 노이즈 분류 모델

In [ ]:
noise_model = AutoModelForSequenceClassification.from_pretrained(
    BASE, num_labels=2, id2label={0: "normal", 1: "noise"}, label2id={"normal": 0, "noise": 1})
# 노이즈가 적어서(약 4%) 학습에서 노이즈를 더 자주 보게 3배로 늘림
train_noise = pd.concat([train, train[train.is_noise == 1], train[train.is_noise == 1]]).sample(frac=1, random_state=0)
Trainer(model=noise_model, args=args("/content/out_noise", 3), train_dataset=to_ds(train_noise, "is_noise"),
        data_collator=collator).train()
p = predict_probs(noise_model, test_text.text.tolist())
test_text["reliability"] = p[:, 0]                     # 정상일 확률

## ② 부정 유형 분류 모델 (멀티라벨)

In [ ]:
type_model = AutoModelForSequenceClassification.from_pretrained(
    BASE, num_labels=len(TYPES), problem_type="multi_label_classification",
    id2label={i: t for i, t in enumerate(TYPES)}, label2id={t: i for i, t in enumerate(TYPES)})
Trainer(model=type_model, args=args("/content/out_type", 4), train_dataset=to_ds(train_clean, multi=True),
        data_collator=collator).train()
P_type = predict_probs(type_model, test_clean.text.tolist(), multi=True)

## ③ 비교: 규칙 / TF-IDF+SVM / KLUE-BERT  (같은 평가셋)

In [ ]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import precision_recall_fscore_support, f1_score

def prf(y, p):
    a, b, c, _ = precision_recall_fscore_support(y, p, average="binary", zero_division=0)
    return round(a, 3), round(b, 3), round(c, 3)
vec = lambda: TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=2, sublinear_tf=True)

# 규칙 (분석 서버 rule-v1 과 같은 식)
NOISE_RULES = [r"https?://|blog\.|\.com|협찬|체험단|원고료", r"이벤트 참여|리뷰 ?이벤트|이벤트로 받", r"○○|다른 가게|옆 가게|여기 말고"]
NEG_RULES = {
    "맛": r"식어|짜요|짰어|눅눅|싱거|맛이 덜|불어|퍼졌|질겼|말라|차가웠|밍밍|미지근|딱딱|묽어|달아요|별로였|맛없",
    "가격": r"비싸|가격이 .*올|가격 대비 별로|양에 비해", "서비스": r"불친절|누락|안 주|무성의|응대|불쾌",
    "위생": r"머리카락|끈적|깨져|지저분|벌레|비위생", "대기시간": r"늦게 나|대기가 길|기다렸|웨이팅이 길|오래 걸",
    "분위기": r"시끄|좁아|추웠|더웠|냄새"}

# --- 노이즈
y = test_text.is_noise.values
t = test_text.text
rule_noise = ((t.str.replace(r"[^가-힣]", "", regex=True).str.len() < 2) | t.map(lambda s: any(re.search(p, s) for p in NOISE_RULES))).astype(int)
v = vec(); svm = LinearSVC(class_weight="balanced").fit(v.fit_transform(train.text), train.is_noise)
svm_noise = svm.predict(v.transform(t))
bert_noise = (test_text.reliability < 0.5).astype(int)
bert_noise_03 = (test_text.reliability < 0.3).astype(int)
noise_table = pd.DataFrame([
    {"방법": "규칙 (rule-v1)", **dict(zip(["정밀도", "재현율", "F1"], prf(y, rule_noise)))},
    {"방법": "TF-IDF+SVM", **dict(zip(["정밀도", "재현율", "F1"], prf(y, svm_noise)))},
    {"방법": "KLUE-BERT (신뢰도<0.5)", **dict(zip(["정밀도", "재현율", "F1"], prf(y, bert_noise)))},
    {"방법": "KLUE-BERT (신뢰도<0.3, 설계 기준)", **dict(zip(["정밀도", "재현율", "F1"], prf(y, bert_noise_03)))},
    {"방법": "규칙 + KLUE-BERT(0.3) (시스템 파이프라인)", **dict(zip(["정밀도", "재현율", "F1"], prf(y, rule_noise | bert_noise_03)))},
])
print("노이즈 분류 — 평가", len(test_text), "건 중 노이즈", int(y.sum()))
display(noise_table)

# --- 부정 유형
Y = test_clean[TYPES].values
rule_types = np.column_stack([test_clean.text.map(lambda s: int(bool(re.search(NEG_RULES[k], s)))) for k in TYPES])
v2 = vec(); X2 = v2.fit_transform(train_clean.text)
svm_types = np.column_stack([LinearSVC(class_weight="balanced").fit(X2, train_clean[k]).predict(v2.transform(test_clean.text)) for k in TYPES])
bert_types = (P_type >= 0.5).astype(int)
rows = []
for name, P in [("규칙 (rule-v1) ※평가셋 보고 만든 규칙", rule_types), ("TF-IDF+SVM", svm_types), ("KLUE-BERT", bert_types)]:
    row = {"방법": name, "micro-F1": round(f1_score(Y, P, average="micro"), 3),
           "macro-F1": round(f1_score(Y, P, average="macro", zero_division=0), 3),
           "부정 여부 F1": prf((Y.sum(1) > 0).astype(int), (P.sum(1) > 0).astype(int))[2]}
    row.update({k: round(f1_score(Y[:, i], P[:, i], zero_division=0), 2) for i, k in enumerate(TYPES)})
    rows.append(row)
type_table = pd.DataFrame(rows)
print("부정 유형 분류 — 평가 정상 리뷰", len(test_clean), "건 중 부정", int((Y.sum(1) > 0).sum()))
display(type_table)

In [ ]:
# 4. BERT가 틀린 리뷰 (오류 분석용)
wrong_noise = test_text[(test_text.reliability < 0.3).astype(int) != test_text.is_noise][["text", "is_noise", "reliability"]]
print("노이즈 모델이 틀린 리뷰", len(wrong_noise)); display(wrong_noise.head(10))
wrong_type = test_clean[(bert_types != Y).any(axis=1)].copy()
wrong_type["정답"] = [", ".join(k for k, v in zip(TYPES, r) if v) for r in Y[(bert_types != Y).any(axis=1)]]
wrong_type["예측"] = [", ".join(k for k, v in zip(TYPES, r) if v) for r in bert_types[(bert_types != Y).any(axis=1)]]
print("유형 모델이 틀린 리뷰", len(wrong_type)); display(wrong_type[["text", "정답", "예측"]].head(15))

In [ ]:
# 5. 모델 저장 → models.zip 내려받기 (약 800MB). 분석 서버의 analysis/models/ 에 풀면
#    analysis/models/noise/config.json, analysis/models/negtype/config.json 이 생김
import json
for name, m in [("noise", noise_model), ("negtype", type_model)]:
    path = f"/content/models/{name}"
    m.save_pretrained(path); tok.save_pretrained(path)
    json.dump({"base_model": BASE, "max_len": MAX_LEN, "task": name, "types": TYPES,
               "noise_table": noise_table.to_dict("records"), "type_table": type_table.to_dict("records")},
              open(f"{path}/training_info.json", "w"), ensure_ascii=False, indent=1)
!cd /content && zip -qr models.zip models && ls -lh models.zip
files.download("/content/models.zip")